# 05 — Final Evaluation & Ablation Study
**DeepLense GSoC 2026 — Pallab Mondal**

This notebook compares:
| Model | Description |
|---|---|
| ResNet-18 baseline | No physics, no ViT |
| ViT baseline | No physics |
| LensPINN (ours) | Physics-informed ViT + CNN decoder |
| HEALSwin (ours) | Physics + Swin Transformer + HEALPix |

Metrics: AUC (micro), AUC (macro), F1 (macro), Accuracy

Fill in the checkpoint paths and run all cells to reproduce the ablation table.

In [ ]:
import sys, os
from pathlib import Path

MY_WORK = Path(os.getcwd()).parent
sys.path.insert(0, str(MY_WORK))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from config import (
    MODEL_I_TEST, CLASS_NAMES, IMAGE_SIZE, BATCH_SIZE, CHECKPOINTS_DIR, PLOTS_DIR
)
from utils.data_loader import DeepLenseDataset, get_val_transform
from utils.metrics     import evaluate_model
from models.lens_pinn  import LensPINN
from models.heal_swin  import HEALSwin
from models.classifier import DeepLenseClassifier, ResNetBaseline
from torch.utils.data  import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Define models and checkpoints to compare ──────────────────────────────────
# Fill in actual checkpoint paths after training each model

CONFIGS = [
    {
        'name': 'ResNet-18 (baseline)',
        'model': ResNetBaseline(num_classes=3),
        'ckpt': CHECKPOINTS_DIR / 'resnet_baseline_model_i_best.pt',
    },
    {
        'name': 'ViT (no physics)',
        'model': DeepLenseClassifier(num_classes=3, in_chans=1),
        'ckpt': CHECKPOINTS_DIR / 'vit_baseline_model_i_best.pt',
    },
    {
        'name': 'LensPINN (ours)',
        'model': LensPINN(num_classes=3),
        'ckpt': CHECKPOINTS_DIR / 'lens_pinn_model_i_best.pt',
    },
    {
        'name': 'HEALSwin (ours)',
        'model': HEALSwin(num_classes=3),
        'ckpt': CHECKPOINTS_DIR / 'heal_swin_model_i_best.pt',
    },
]

print(f'{len(CONFIGS)} models configured for comparison.')

In [ ]:
# ── Prepare test set ──────────────────────────────────────────────────────────
test_ds = DeepLenseDataset(
    root=MODEL_I_TEST,
    class_names=CLASS_NAMES[:3],
    transform=get_val_transform(IMAGE_SIZE),
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available(),
)

print(f'Test samples: {len(test_ds)}')

In [ ]:
# ── Run comparison ────────────────────────────────────────────────────────────
rows = []

for cfg in CONFIGS:
    model = cfg['model'].to(device)
    ckpt_path = cfg['ckpt']

    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt.get('model_state_dict', ckpt))
        print(f"[✓] Loaded: {cfg['name']}")
    else:
        print(f"[!] No checkpoint for {cfg['name']} — using random weights")

    res = evaluate_model(model, test_loader, device, class_names=CLASS_NAMES[:3], verbose=False)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    rows.append({
        'Model':      cfg['name'],
        'AUC micro':  res['auc_micro'],
        'AUC macro':  res['auc_macro'],
        'F1 macro':   res['f1_macro'],
        'Accuracy':   res['accuracy'],
        'Params (M)': n_params / 1e6,
    })

df = pd.DataFrame(rows).set_index('Model')
df = df.sort_values('AUC macro', ascending=False)

# Display
print('\n' + '='*65)
print('  ABLATION STUDY — Model Comparison (Model I test set)')
print('='*65)
print(df.to_string(float_format='{:.4f}'.format))
print('='*65)

In [ ]:
# ── Plot comparison ───────────────────────────────────────────────────────────
metrics = ['AUC micro', 'AUC macro', 'F1 macro', 'Accuracy']
x = np.arange(len(df))
width = 0.18
colors = ['#4C9BE8', '#50C878', '#E86C4C', '#9B4CE8']

fig, ax = plt.subplots(figsize=(12, 5))

for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = df[metric].values
    bars = ax.bar(x + i * width - 1.5 * width, vals, width, label=metric, color=color, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set(title='Ablation Study — Model Comparison on Model I',
       ylabel='Score', ylim=(0, 1.08),
       xticks=x, xticklabels=df.index)
ax.tick_params(axis='x', labelsize=9)
ax.legend(loc='upper right')
plt.setp(ax.get_xticklabels(), rotation=15, ha='right')
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '05_ablation_comparison.png'), dpi=150)
plt.show()

In [ ]:
# ── Export results table as LaTeX for the paper ───────────────────────────────
latex_table = df[['AUC micro', 'AUC macro', 'F1 macro', 'Accuracy']].to_latex(
    float_format='{:.4f}'.format,
    caption='Ablation study on DeepLense Model I test set.',
    label='tab:ablation',
)
print(latex_table)